# RG3 F1: CN7 보조 학습의 조건부 순위 전달

평가는 기존 RG3 RH620(정상601·가스19)/같은7개구간으로 고정. CN7 RH705(정상695·가스10)는 보조 학습으로만 사용하며 평가 개수에 합치지 않는다. 제품별 정상 이웃75 조건부 순위10개로 제품 수준 차이 보정. CN7/RG3 시각 겹침은 보수적으로 제외·검증. CN7 가스가 RG3와 같은 물리 원인임을 보장하지 않음.

CN7 보조 가중 .25/.5/1을 사전 고정. RF/ET2시드 평균 전문가와 기존 RG3만 전문가의 단독/쌍평균을 내부 RG3 F1로 선택. 외부 RG3 구간은 정상순위·분류·가중치·임계값 학습에서 제외. CN7 라벨은 학습용 알려진 자료이고 RG3 평가 라벨은 집계에만 사용. 이미 본 자료 개발용 중첩검증, 새 독립 검증 아님.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/74_product_normalized_joint_gas_modes.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.rsplit('memo={}',1)[0])
joint=d.copy()
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19
rg=joint[joint['product'].eq('RG3')].set_index('TimeStamp').loc[pd.to_datetime(times)].reset_index()
assert np.array_equal(rg.PassOrFail.to_numpy(int),y)
cn=joint[joint['product'].eq('CN7')].copy().reset_index(drop=True)
assert len(cn)==705 and cn.PassOrFail.sum()==10
# Exclude shared timestamps conservatively; product name alone is not independent sensor evidence.
cn=cn[~cn.TimeStamp.astype(str).isin(times)].reset_index(drop=True)
assert len(cn)==705
cn_y=cn.PassOrFail.to_numpy(int)
state=cn[['Hopper_Temperature','Barrel_Temperature_2','Barrel_Temperature_5','Cushion_Position','Plasticizing_Time']].copy()
state['pressure_switch_gap']=cn.Max_Injection_Pressure-cn.Max_Switch_Over_Pressure
stdcols=[v+'__past10_std' for v in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
base=pd.concat([state,cn[stdcols]],axis=1).replace([np.inf,-np.inf],np.nan)
normal=np.flatnonzero(cn_y==0)
im=SimpleImputer(strategy='median',keep_empty_features=True).fit(base.iloc[normal])
base=im.transform(base);sc=RobustScaler().fit(base[normal]);context=np.clip(sc.transform(base),-8,8)
cn_rank=np.empty_like(base);k=75
for col in range(base.shape[1]):
 other=np.arange(base.shape[1])!=col
 nn=NearestNeighbors(n_neighbors=k+1).fit(context[normal][:,other])
 indices=nn.kneighbors(context[:,other],return_distance=False)
 for row in range(len(cn)):
  ids=indices[row][normal[indices[row]]!=row][:k]
  peers=base[normal[ids],col]
  cn_rank[row,col]=(1+(peers<base[row,col]).sum()+.5*(peers==base[row,col]).sum())/(len(peers)+2)
sources={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
memo={};audit=[]
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=tuple(ref)
 if key not in memo:
  n=sources[key];F=z['features_'+n+'_1'];old=z['scores_'+n][:,1]
  tr=np.r_[F[ref],cn_rank];yy=np.r_[y[ref],cn_y]
  values=[old]
  for weight in [.25,.5,1.]:
   w=np.r_[np.ones(len(ref)),np.full(len(cn),weight)];pred=[]
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
     pred.append(m.fit(tr,yy,sample_weight=w).predict_proba(F)[:,1])
   values.append(np.mean(pred,axis=0))
  memo[key]=np.column_stack(values)
  audit.append(dict(RG3_train=len(ref),RG3_train_gas=int(y[ref].sum()),CN7_train=len(cn),CN7_train_gas=int(cn_y.sum()),RG3_train_blocks=','.join(map(str,np.unique(groups[ref]))),excluded_RG3_blocks=','.join(map(str,np.setdiff1d(blocks,groups[ref])))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
mixes=[np.eye(4)[i] for i in range(4)]+[(np.eye(4)[i]+np.eye(4)[j])/2 for i in range(4) for j in range(i+1,4)]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b);out=predict(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for w in mixes:
  cut,result=cutfit(y[ref],inner@w);candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,out@w))
 best=max(candidates,key=lambda a:a[:3])
 named=[('inner_F1_selected_CN7_support',best[5],best[4])]
 for i,name in enumerate(['RG3_only_reference','CN7_weight025','CN7_weight05','CN7_weight1']):
  cut,_=cutfit(y[ref],inner[:,i]);named.append((name+'_inner_F1_cut',out[:,i],cut))
 for name,score,cut in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(score[j]>=cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(b),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0]))
 (t/'rg3_CN7_support_progress.json').write_text(json.dumps(dict(last_completed_block=int(b),trained_ref_count=len(memo),RG3_eval_gas=19,CN7_auxiliary_gas=int(cn_y.sum()))),encoding='utf-8')
 print('Finished RG3 evaluation block with CN7 support',b,flush=True)
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(t/f'rg3_CN7_support_{name}.csv',index=False,encoding='utf-8-sig')
cache={'labels':y,'groups':groups,'timestamps':times,'CN7_auxiliary_labels':cn_y,'CN7_auxiliary_rank':cn_rank}
for j,(ref,score) in enumerate(memo.items()):cache[f'ref_{j}']=np.array(ref,int);cache[f'scores_{j}']=score
np.savez_compressed(t/'rg3_CN7_support_fold_cache.npz',**cache)
print(m.to_string(index=False))
print('Evaluation remains RG3 normal601/gas19; CN7 auxiliary705 is NOT included in metrics.')


CN7 extra metadata 713 sensor-consistent unique inputs 713 conflicting 0


Finished RG3 evaluation block with CN7 support 1


Finished RG3 evaluation block with CN7 support 2


Finished RG3 evaluation block with CN7 support 3


Finished RG3 evaluation block with CN7 support 4


Finished RG3 evaluation block with CN7 support 5


Finished RG3 evaluation block with CN7 support 6


Finished RG3 evaluation block with CN7 support 7


                          model       AP  TP  FP  FN  TN       F1   recall
     CN7_weight025_inner_F1_cut 0.052956   5  48  14 553 0.138889 0.263158
      CN7_weight05_inner_F1_cut 0.046993   3  55  16 546 0.077922 0.157895
       CN7_weight1_inner_F1_cut 0.038882   3  85  16 516 0.056075 0.157895
RG3_only_reference_inner_F1_cut 0.076694   5  35  14 566 0.169492 0.263158
  inner_F1_selected_CN7_support 0.067327   6  42  13 559 0.179104 0.315789
Evaluation remains RG3 normal601/gas19; CN7 auxiliary705 is NOT included in metrics.
